In [1]:
print("hi")

hi


```mermaid
graph LR
    A[CineBot Agent] -->|MCPAdapter| B[(MCP Server)]
    B --> C[check_showtimes]
    B --> D[cancel_booking]
    B --> E[get_seat_map]
```

In [3]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
assert os.environ.get("OPENAI_API_KEY") is not None, "OPENAI_API_KEY is not set in the environment variables."

In [6]:
import langchain.mcp

/var/folders/gd/gpz62qhd2k59xc_wfb3qp1f80000gn/T/ipykernel_62893/1520507502.py:1: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  import langchain.mcp


# MCP Adapter in LangChain

In [ ]:
import langchain.mcp


In [17]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_mcp_server.py


In [8]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

In [20]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("/Users/mayank/Github Repos/Agentic 3.0/Live-Class-2026/Weekend 14 - Oct 3 & Oct 4/MCP in Langchain/cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


In [21]:
from langchain.agents import create_agent

# Call a tool without any AI or Agent


In [ ]:
tool_call_result=''
async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well

    try:
        cinebot_agent = create_agent(model='openai:gpt-5-mini',tools=tools)
        result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        tool_call_result = result
        print("Agent invocation result:", result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_3852a2a6-2515-4130-9e62-9df2a749b46b'}]
Agent invocation result: {'messages': [HumanMessage(content='What are the showtimes for Interstellar?', additional_kwargs={}, response_metadata={}, id='a15714b5-9d35-4814-834b-c3eaa4fa5d09'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 194, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUkUjDLPZqp0DPTdPmSHRLm4FERoq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id

In [28]:
from rich import print as rprint
rprint("[bold green]Direct tool invocation result:[/bold green]", tool_call_result)

Direct tool invocation result:
{
    'messages': [
        HumanMessage(
            content='What are the showtimes for Interstellar?',
            additional_kwargs={},
            response_metadata={},
            id='a15714b5-9d35-4814-834b-c3eaa4fa5d09'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 26,
                    'prompt_tokens': 194,
                    'total_tokens': 220,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUkUjDLPZqp0DPTdPmSHRLm4FERoq',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0ffc1-95e1-7f53-a99f-dde045325988-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'call_2wcUW4Tso7jtZ0yrbjfZEfTW',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 194,
                'output_tokens': 26,
                'total_tokens': 220,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content=[
                {'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_f02f1757-50b6-436e-95d8-93686c1dcc53'}
            ],
            name='check_showtimes',
            id='7623b573-b8f6-437e-a3f3-0c981cee6b51',
            tool_call_id='call_2wcUW4Tso7jtZ0yrbjfZEfTW',
            artifact={'structured_content': {'result': '7:00 PM and 10:15 PM'}}
        ),
        AIMessage(
            content='Interstellar is showing at 7:00 PM and 10:15 PM. \n\nWould you like to book tickets or see the
seat map for either showing?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 170,
                    'prompt_tokens': 236,
                    'total_tokens': 406,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUkUki86WrPYOJ7U0fqBX6HYJKTyf',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logp

In [31]:
tool_call_result['messages'][-1]

AIMessage(content='Interstellar is showing at 7:00 PM and 10:15 PM. \n\nWould you like to book tickets or see the seat map for either showing?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 236, 'total_tokens': 406, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUkUki86WrPYOJ7U0fqBX6HYJKTyf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ffc1-9d48-7aa0-9fa6-c66837d8b725-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 170, 'total_tokens': 406, 'input_token_details': {'audi

# 5. Tool Output: Multimodal Content Blocks

An MCP tool result isn't limited to text. `langchain.mcp` converts whatever the server sends —
text, images, embedded files — into standard LangChain **content blocks**, so a model sees a
uniform shape regardless of what kind of server produced it.

| MCP content type | Converts to |
|---|---|
| `TextContent` | text content block |
| `ImageContent` | image content block (base64 + mime type) |
| `ResourceLink` (image mime type) | image content block (by URL) |
| `ResourceLink` (other) | file content block |
| `EmbeddedResource` (text) | text content block |
| `EmbeddedResource` (blob) | image or file content block, by mime type |
| `AudioContent` | **not yet supported** — raises `NotImplementedError` |

# Structured Output 

In [37]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    print("Text Content (what my model reads)",message.content)
    print("Structured Data (what my code can use if required)",message.artifact)
    print(message.artifact['structured_content']['available_rows'])  # Accessing structured data directly

Text Content (what my model reads) [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_b4f191a2-de81-4303-a2fb-c47c3cc851d9'}]
Structured Data (what my code can use if required) {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
['A', 'B', 'C']
